# This notebook is used to generate raw markercloud and associated CoP/GRF paired data

In [ ]:
import numpy as np
import ezc3d
import nimblephysics as nimble
import pandas as pd
import matplotlib.pyplot as plt

def load_data_b3d(b3d_path):
    """
    This function loads marker and ground reaction force data from a B3D file and returns a list of dictionaries
    containing marker cloud, cop, grf, subject mass, and frequency data for each trial.
    
    Args:
        b3d_path: Path to the B3D file
    """
    # Load subject from B3D file
    subject = nimble.biomechanics.SubjectOnDisk(b3d_path)
    num_trials = subject.getNumTrials()

    trial_list = []

    for trial_num in range(num_trials):
        # Load frames from a trial
        frames = subject.readFrames(
            trial=trial_num,
            startFrame=0,
            numFramesToRead=subject.getTrialLength(trial_num),
            includeProcessingPasses=True  # This is the correct parameter if you need processing passes
        )

        num_force_plates = subject.getNumForcePlates(trial=trial_num)
        cop = np.zeros((len(frames), num_force_plates, 3))  # Center of pressure
        grf = np.zeros((len(frames), num_force_plates, 3))  # Ground reaction forces
        num_markers = max(len(frame.markerObservations) for frame in frames)
        marker_clouds = np.zeros((len(frames), num_markers, 3))
        for i, frame in enumerate(frames):
            cop[i] = frame.rawForcePlateCenterOfPressures
            grf[i] = frame.rawForcePlateForces
            observed_markers = [obs[1] for obs in frame.markerObservations]
            num_observed = len(observed_markers)
            if num_observed > 0:
                marker_clouds[i, :num_observed, :] = np.array(observed_markers)

        cop = cop[:, :, [0, 2, 1]]
        grf = grf[:, :, [0, 2, 1]]
        marker_clouds = marker_clouds[:, :, [0, 2, 1]]

        trial_dict = {
            "marker_clouds": marker_clouds,
            "cop": cop,
            "grf": grf,
            "subject_mass": subject.getMassKg(),
            "freq": 1 / subject.getTrialTimestep(trial_num)
        }
        trial_list.append(trial_dict)

    return trial_list

In [ ]:
b3d_path = "/home/darren/Desktop/grf_estimation/data/AddBiomechanicsDataset/train/With_Arm/Han2023_Formatted_With_Arm/s006_split1/s006_split1.b3d"

trial_list = load_data_b3d(b3d_path)


In [32]:
import pickle
import gzip

# Save
with gzip.open('Han2023_s006_split1_train.pkl.gz', 'wb') as f:
    pickle.dump(trial_list, f, protocol=pickle.HIGHEST_PROTOCOL)

# Load
with gzip.open('Han2023_s006_split1_train.pkl.gz', 'rb') as f:
    trial_list = pickle.load(f)